##### Lab 10: Automated Multi-Format Sales Report Pipeline and Delivery


##### Objective

To design and implement an automated reporting pipeline that:

- Extracts sales data from a dataset
- Transforms the data and calculates revenue
- Generates reports in multiple formats (Excel, HTML, PDF)
- Saves reports with the current date in a **reports folder**
- Optionally sends the generated report via email using SMTP

In [1]:
# Import libraries for data processing
import pandas as pd

# Import library for folder and file operations
import os

# Import library for working with dates
from datetime import datetime

# Import library for creating HTML templates
from jinja2 import Template

# Import library for generating PDF reports
from fpdf import FPDF

# Import libraries for sending email
import smtplib
from email.message import EmailMessage

In [2]:
# Load the dataset from CSV file
df = pd.read_csv("sales_raw.csv")

# Display first few rows to verify data
df.head()

,order_id,customer_id,product,quantity,price,order_date,city
0,1001,C01,Laptop,1,55000.0,2024-01-05,Kathmandu
1,1002,C02,Phone,2,25000.0,2024-01-06,Pokhara
2,1003,C01,Headphones,1,3000.0,2024-01-07,Kathmandu
3,1004,C03,Tablet,1,35000.0,2024-01-08,Lalitpur
4,1005,C04,Camera,1,80000.0,2024-01-09,Biratnagar


In [3]:
# Convert order_date column to datetime format
df["order_date"] = pd.to_datetime(df["order_date"])

# Extract month from the date
df["Month"] = df["order_date"].dt.strftime("%b")

# Rename columns to match required fields
df = df.rename(columns={
    "product": "Product",
    "quantity": "Units Sold",
    "price": "Unit Price",
    "city": "Region"
})

# Display updated dataset
df.head()

,order_id,customer_id,Product,Units Sold,Unit Price,order_date,Region,Month
0,1001,C01,Laptop,1,55000.0,2024-01-05,Kathmandu,Jan
1,1002,C02,Phone,2,25000.0,2024-01-06,Pokhara,Jan
2,1003,C01,Headphones,1,3000.0,2024-01-07,Kathmandu,Jan
3,1004,C03,Tablet,1,35000.0,2024-01-08,Lalitpur,Jan
4,1005,C04,Camera,1,80000.0,2024-01-09,Biratnagar,Jan


In [4]:
# Calculate revenue using formula:
# Revenue = Units Sold × Unit Price
df["Revenue"] = df["Units Sold"] * df["Unit Price"]

# Display updated data
df.head()

,order_id,customer_id,Product,Units Sold,Unit Price,order_date,Region,Month,Revenue
0,1001,C01,Laptop,1,55000.0,2024-01-05,Kathmandu,Jan,55000.0
1,1002,C02,Phone,2,25000.0,2024-01-06,Pokhara,Jan,50000.0
2,1003,C01,Headphones,1,3000.0,2024-01-07,Kathmandu,Jan,3000.0
3,1004,C03,Tablet,1,35000.0,2024-01-08,Lalitpur,Jan,35000.0
4,1005,C04,Camera,1,80000.0,2024-01-09,Biratnagar,Jan,80000.0


In [5]:
# Group data by product and calculate summary statistics
product_summary = df.groupby("Product").agg(
    Total_Units=("Units Sold","sum"),
    Total_Revenue=("Revenue","sum"),
    Avg_Price=("Unit Price","mean")
).reset_index()

# Calculate total revenue
total_revenue = product_summary["Total_Revenue"].sum()

# Calculate revenue share percentage
product_summary["Revenue Share %"] = (
    product_summary["Total_Revenue"] / total_revenue * 100
)

# Display product summary
product_summary

,Product,Total_Units,Total_Revenue,Avg_Price,Revenue Share %
0,Camera,1,80000.0,80000.0,22.662890
1,Charger,3,0.0,NaN,0.000000
2,Headphones,1,3000.0,3000.0,0.849858
3,Laptop,2,110000.0,55000.0,31.161473
4,Phone,5,125000.0,25000.0,35.410765
5,Tablet,1,35000.0,35000.0,9.915014


In [6]:
# Calculate total revenue for each month
monthly_revenue = df.groupby("Month")["Revenue"].sum().reset_index()

# Display monthly revenue
monthly_revenue

,Month,Revenue
0,Jan,353000.0


In [7]:
# Create a folder named "reports" if it does not exist
os.makedirs("reports", exist_ok=True)

# Get today's date for filename
today = datetime.now().strftime("%Y-%m-%d")

In [8]:
# Define Excel file path
excel_file = f"reports/sales_report_{today}.xlsx"

# Write data into Excel with multiple sheets
with pd.ExcelWriter(excel_file) as writer:
    
    # Write product summary sheet
    product_summary.to_excel(writer, sheet_name="Product Summary", index=False)
    
    # Write monthly revenue sheet
    monthly_revenue.to_excel(writer, sheet_name="Monthly Revenue", index=False)

excel_file

'reports/sales_report_2026-03-14.xlsx'

In [9]:
from jinja2 import Template

# Folder and date for saving
folder = "reports"
date = datetime.now().strftime("%Y-%m-%d")

# HTML template with styles and alternating rows
html_template = """
<html>
<head>
<style>
body {font-family: Arial, sans-serif;}
h2 {color: #333;}
table {border-collapse: collapse; width: 80%; margin-bottom: 20px;}
th, td {border: 1px solid #333; padding: 8px; text-align: center;}
th {background-color: #4CAF50; color: white;}
tr:nth-child(even) {background-color: #f2f2f2;}
</style>
</head>

<body>

<h2>Sales Summary Report</h2>
<table>
<tr>
  <th>Product</th>
  <th>Total Units</th>
  <th>Total Revenue</th>
  <th>Average Price</th>
  <th>Revenue Share %</th>
</tr>
{% for row in summary %}
<tr>
  <td>{{ row.Product }}</td>
  <td>{{ row.Total_Units }}</td>
  <td>{{ row.Total_Revenue }}</td>
  <td>{{ "%.2f"|format(row.Avg_Price) }}</td>
  <td>{{ "%.2f"|format(row["Revenue Share %"]) }}%</td>
</tr>
{% endfor %}
</table>

<h2>Monthly Revenue</h2>
<table>
<tr>
  <th>Month</th>
  <th>Total Revenue</th>
</tr>
{% for row in monthly %}
<tr>
  <td>{{ row.Month }}</td>
  <td>{{ row.Revenue }}</td>
</tr>
{% endfor %}
</table>

<h3>Total Revenue: {{ total }}</h3>

</body>
</html>
"""

# Create template object
template = Template(html_template)

# Render the template
html_content = template.render(
    summary=product_summary.to_dict(orient="records"),
    monthly=monthly_revenue.to_dict(orient="records"),
    total=total_revenue
)

# Save HTML file
html_file = f"{folder}/sales_report_{date}.html"

with open(html_file, "w") as f:
    f.write(html_content)

html_file

'reports/sales_report_2026-03-14.html'

In [21]:
from fpdf import FPDF
from fpdf.enums import XPos, YPos
import pandas as pd

# Sample summary data (replace with your real summary DataFrame)
summary_data = {
    "Product": ["Laptop", "Phone", "Tablet"],
    "Total Units": [120, 250, 180],
    "Total Revenue": [660000, 700000, 351000],
    "Avg Price": [5500, 2800, 1950],
    "Revenue Share (%)": [30.5, 32.3, 16.2]
}
summary = pd.DataFrame(summary_data)

# Create PDF
pdf = FPDF()
pdf.add_page()

# ===== TITLE =====
pdf.set_font("Helvetica", "B", 16)   # use Helvetica instead of Arial
pdf.cell(0, 10, "Sales Report", border=0, align="C")  # no ln parameter
pdf.ln(15)  # move to next line

# ===== TABLE HEADER =====
pdf.set_font("Helvetica", "B", 12)
headers = ["Product", "Total Units", "Total Revenue", "Avg Price", "Revenue Share (%)"]
col_widths = [40, 30, 40, 30, 40]

for i, header in enumerate(headers):
    pdf.cell(col_widths[i], 10, header, border=1, align="C")
pdf.ln()

# ===== TABLE ROWS =====
pdf.set_font("Helvetica", "", 12)
fill = False  # for alternating row colors

for index, row in summary.iterrows():
    if fill:
        pdf.set_fill_color(220, 220, 220)  # light gray
    else:
        pdf.set_fill_color(255, 255, 255)  # white

    pdf.cell(col_widths[0], 10, str(row["Product"]), border=1, align="C", fill=True)
    pdf.cell(col_widths[1], 10, str(row["Total Units"]), border=1, align="C", fill=True)
    pdf.cell(col_widths[2], 10, f"{row['Total Revenue']:,}", border=1, align="C", fill=True)
    pdf.cell(col_widths[3], 10, f"{row['Avg Price']:,}", border=1, align="C", fill=True)
    pdf.cell(col_widths[4], 10, f"{row['Revenue Share (%)']:.1f}%", border=1, align="C", fill=True)
    pdf.ln()
    fill = not fill  # alternate row color

# Save PDF
pdf.output("sales_report.pdf")
print("PDF report generated as sales_report.pdf")

PDF report generated as sales_report.pdf


In [15]:
#Generating PDF report
from fpdf import FPDF, XPos, YPos

pdf = FPDF()
pdf.add_page()

# Title
pdf.set_font("Helvetica","B",16)
pdf.cell(0,10,"Sales Report with Monthly Revenue Chart",0,new_x=XPos.LMARGIN,new_y=YPos.NEXT,align="C")

# Table header
pdf.set_font("Helvetica","B",12)
headers = ["Product","Total Units","Total Revenue","Avg Price","Revenue Share %"]
for h in headers:
    pdf.cell(38,10,h,1)
pdf.ln()

# Table rows
pdf.set_font("Helvetica","",12)
for i,row in product_summary.iterrows():
    pdf.cell(38,10,str(row["Product"]),1)
    pdf.cell(38,10,str(row["Total_Units"]),1)
    pdf.cell(38,10,str(row["Total_Revenue"]),1)
    pdf.cell(38,10,f"{row['Avg_Price']:.2f}",1)
    pdf.cell(38,10,f"{row['Revenue Share %']:.2f}%",1)
    pdf.ln()

# Monthly revenue bar chart
pdf.set_font("Helvetica","",10)
max_revenue = monthly_revenue["Revenue"].max()
chart_width = 150
x_start = 30
y_start = pdf.get_y() + 10

for idx, row in monthly_revenue.iterrows():
    bar_length = (row["Revenue"]/max_revenue) * chart_width
    pdf.set_fill_color(0,102,204)
    pdf.rect(x_start, y_start + idx*12, bar_length, 8, 'FD')
    pdf.text(x_start + bar_length + 2, y_start + idx*12 + 6, f"{row.Month}: {row.Revenue}")

# Save PDF
pdf_file_chart = f"reports/sales_report_{today}_chart.pdf"
pdf.output(pdf_file_chart)
pdf_file_chart

'reports/sales_report_2026-03-14_chart.pdf'

In [16]:
# Function to send report via email
def send_email(file):

    sender = "your_email@gmail.com"
    password = "your_app_password"
    receiver = "receiver@gmail.com"

    # Create email message
    msg = EmailMessage()
    
    msg["Subject"] = "Automated Sales Report"
    msg["From"] = sender
    msg["To"] = receiver

    msg.set_content("Attached is the automated sales report.")

    # Attach file
    with open(file,"rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="application",
            subtype="octet-stream",
            filename=file
        )

    # Send email using Gmail SMTP
    with smtplib.SMTP_SSL("smtp.gmail.com",465) as smtp:
        smtp.login(sender,password)
        smtp.send_message(msg)

# Example usage
# send_email(pdf_file)

Q1: Explain the role of each step in the automated report pipeline: Extract → Transform → Generate → Deliver

1. **Extract**  
   - Load the raw sales dataset (CSV, database, etc.) into the pipeline.
   - Collects all necessary information (e.g., product, units sold, price, date, region) to perform reporting.

2. **Transform**  
   - Clean and process the raw data.
   - Examples: converting dates, renaming columns, calculating revenue (`Units Sold × Unit Price`), aggregating totals.
   - Prepares data in a consistent format for reporting.

3. **Generate**  
   - Produce reports in multiple formats (Excel, HTML, PDF) from the processed data.
   - Includes styling, tables, summaries, charts (if needed).
   - Ensures reports are consistent and readable for decision-makers.

4. **Deliver**  
   - Save generated reports in the designated folder with date-based filenames.
   - Optionally send the reports via email using SMTP for automated delivery.

 Q2: How does the pipeline ensure that Excel, HTML, and PDF reports are consistent in content

**Answer:**

- **Single Source of Truth:** All reports (Excel, HTML, PDF) are generated from the same processed DataFrames: `product_summary` and `monthly_revenue`.  
- **Centralized Calculations:** Revenue, Total Units, Average Price, and Revenue Share are calculated **once** in the Transform stage, so all reports use identical numbers.  
- **Automated Generation:** Each report pulls directly from these DataFrames, eliminating manual errors.  
- **Result:** All formats display the same data, ensuring complete consistency across Excel, HTML, and PDF.

Q) Modify the pipeline to include a bar chart of monthly revenue in the PDF report.



In [22]:
from fpdf import FPDF, XPos, YPos

pdf = FPDF()
pdf.add_page()

# Title
pdf.set_font("Helvetica","B",16)
pdf.cell(0,10,"Sales Report with Monthly Revenue Chart",0,new_x=XPos.LMARGIN,new_y=YPos.NEXT,align="C")

# Table header
pdf.set_font("Helvetica","B",12)
headers = ["Product","Total Units","Total Revenue","Avg Price","Revenue Share %"]
for h in headers:
    pdf.cell(38,10,h,1)
pdf.ln()

# Table rows
pdf.set_font("Helvetica","",12)
for i,row in product_summary.iterrows():
    pdf.cell(38,10,str(row["Product"]),1)
    pdf.cell(38,10,str(row["Total_Units"]),1)
    pdf.cell(38,10,str(row["Total_Revenue"]),1)
    pdf.cell(38,10,f"{row['Avg_Price']:.2f}",1)
    pdf.cell(38,10,f"{row['Revenue Share %']:.2f}%",1)
    pdf.ln()

# Monthly revenue bar chart
pdf.set_font("Helvetica","",10)
max_revenue = monthly_revenue["Revenue"].max()
chart_width = 150
x_start = 30
y_start = pdf.get_y() + 10

for idx, row in monthly_revenue.iterrows():
    bar_length = (row["Revenue"]/max_revenue) * chart_width
    pdf.set_fill_color(0,102,204)
    pdf.rect(x_start, y_start + idx*12, bar_length, 8, 'FD')
    pdf.text(x_start + bar_length + 2, y_start + idx*12 + 6, f"{row.Month}: {row.Revenue}")

# Save PDF
pdf_file_chart = f"reports/sales_report_{today}_chart.pdf"
pdf.output(pdf_file_chart)
pdf_file_chart

'reports/sales_report_2026-03-14_chart.pdf'

 Q4: How would you secure email credentials when deploying this pipeline in production

**Answer:**

- **Never hard-code credentials** directly in the Python script.  
- **Use environment variables** or a `.env` file:  
  ```python
  import os
  EMAIL_USER = os.getenv("EMAIL_USER")
  EMAIL_PASS = os.getenv("EMAIL_PASS")
  ```
- **Use App Passwords** (e.g., for Gmail) instead of your main account password.  
- **Limit permissions** on the server or environment where the credentials are stored.  
- Optionally, use **secret managers** (AWS Secrets Manager, Azure Key Vault, etc.) to fetch credentials securely at runtime.  
- Benefits: Protects sensitive data, reduces risk if code is shared or deployed.

In [23]:
import smtplib
from email.message import EmailMessage
import os

def send_email(file):
    sender = os.getenv("EMAIL_USER")
    password = os.getenv("EMAIL_PASS")
    receiver = "receiver@example.com"

    msg = EmailMessage()
    msg["Subject"] = "Automated Sales Report"
    msg["From"] = sender
    msg["To"] = receiver
    msg.set_content("Attached is the automated sales report.")

    with open(file,"rb") as f:
        msg.add_attachment(f.read(),
                           maintype="application",
                           subtype="octet-stream",
                           filename=file)

    with smtplib.SMTP_SSL("smtp.gmail.com",465) as smtp:
        smtp.login(sender,password)
        smtp.send_message(msg)

# Example usage:
# send_email(pdf_file_chart)

##### Conclusion

In this lab, an automated multi-format sales reporting pipeline was implemented.  
The system extracts sales data, transforms it to calculate revenue, generates reports in Excel, HTML, and PDF formats, and optionally sends the report via email.

This type of automation is commonly used in business analytics and reporting systems.